# Unit V — Guided Supervised Learning Project
**Pharmakaksha · Learn · Grow · Achieve** — BP703T AI in Clinical Applications (B.Pharm Sem VII, PCI NEP 2020)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajreddys/learn-pharmakaksha-site/blob/main/content/BP703T%20AI%20in%20Clinical%20Applications%20%28Theory%29/BP703T_Unit5_Guided_Project.ipynb)

Two complete worked projects following the syllabus steps: **A** on real public data (Pima diabetes, logistic regression) and **B** on fictional data (medication adherence, linear regression), then a template for your own project and lab exercises with solutions. Data: `pima_diabetes.csv`, `adherence_pdc.csv`.

## Using learn.pharmakaksha.com
- **First run is slow.** Python starts inside your browser; the first cell can take about 10 seconds. Later cells are fast.
- **Kernel indicator.** The circle at the top right is filled while Python is busy. Wait for it to clear.
- **Stuck?** Use *Kernel → Restart Kernel*, then run the cells again from the top.
- **Saving.** Your changes are saved in this browser only. Use *File → Download* to keep a copy.
- **Start over.** Delete your copy from the file list on the left and reopen the notebook from the course folder.

In [ ]:
import os
import numpy as np
import pandas as pd
BASE = "https://raw.githubusercontent.com/ajreddys/learn-pharmakaksha-site/main/content/BP703T%20AI%20in%20Clinical%20Applications%20%28Theory%29/"

def load(name):
    """Read a practice CSV: local copy first, then the course's GitHub folder."""
    if os.path.exists(name):
        return pd.read_csv(name)
    try:
        return pd.read_csv(BASE + name)
    except Exception:
        from pyodide.http import open_url   # inside JupyterLite
        return pd.read_csv(open_url(BASE + name))

# Project A — real public data: predicting diabetes (logistic regression)
The Pima Indians Diabetes dataset: 768 women aged 21+ of Pima (Akimel O'odham) heritage, near Phoenix, USA, originally collected by the US National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK) and widely shared through the UCI Machine Learning Repository.

### Step 1 · Define the problem
*Can routine measurements predict which women have diabetes, so they can be prioritised for confirmatory testing?* Outcome: `outcome` (1 = diabetes). Model: logistic regression.

In [ ]:
pima = load("pima_diabetes.csv")
print(pima.shape)
pima.head()

### Step 2–3 · Choose data, predictors and outcome; explore

In [ ]:
pima.describe().round(1)

In [ ]:
zero_cols = ["glucose", "blood_pressure", "skin_thickness", "insulin", "bmi"]
print("Share recorded as 0 (physiologically impossible = missing):")
print((pima[zero_cols] == 0).mean().round(3))

### Step 4 · Preprocess
Replace impossible zeros with NaN. Drop insulin and skin thickness (too much missing). Impute the rest with the **training-set** median inside a Pipeline, so no information leaks from the test set.

In [ ]:
for c in zero_cols:
    pima[c] = pima[c].replace(0, np.nan)
features = ["pregnancies", "glucose", "blood_pressure", "bmi", "diabetes_pedigree", "age"]
X, y = pima[features], pima["outcome"]
print("Diabetes prevalence:", round(y.mean(), 3))

### Step 5 · Fit

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, mean_squared_error, r2_score
import matplotlib.pyplot as plt

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)
p = model.predict_proba(X_te)[:, 1]

### Step 6 · Evaluate

In [ ]:
glucose_only = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression()).fit(X_tr[["glucose"]], y_tr)
p_g = glucose_only.predict_proba(X_te[["glucose"]])[:, 1]
print("Test cases:", len(y_te), "| with diabetes:", int(y_te.sum()))
print("AUC, six predictors:", round(roc_auc_score(y_te, p), 3))
print("AUC, glucose only:  ", round(roc_auc_score(y_te, p_g), 3))
for pr, lab in [(p, "six predictors"), (p_g, "glucose only")]:
    f, t, _ = roc_curve(y_te, pr); plt.plot(f, t, label=lab)
plt.plot([0, 1], [0, 1], "--", color="grey"); plt.xlabel("1 − specificity"); plt.ylabel("Sensitivity"); plt.legend(); plt.show()

In [ ]:
def evaluate(y_true, prob, threshold):
    tn, fp, fn, tp = confusion_matrix(y_true, (prob >= threshold).astype(int)).ravel()
    return {"threshold": threshold, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "accuracy": round((tp + tn) / len(y_true), 3), "sensitivity": round(tp / (tp + fn), 2),
            "specificity": round(tn / (tn + fp), 2), "precision": round(tp / (tp + fp), 2)}
pd.DataFrame([evaluate(y_te, p, 0.5), evaluate(y_te, p, 0.3)])

### Step 7 · Interpret
Refit without scaling so each odds ratio is per natural unit.

In [ ]:
plain = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(C=1e6, max_iter=5000)).fit(X_tr, y_tr)
odds = pd.Series(np.exp(plain[-1].coef_[0]), index=features)
print(odds.round(3))
print("\nGlucose, per 10 mg/dL:", round(odds["glucose"] ** 10, 2))
print("BMI, per 5 kg/m²:     ", round(odds["bmi"] ** 5, 2))

### Step 8 · Bias and ethics

In [ ]:
te = X_te.copy(); te["y"] = y_te.values; te["p"] = p
for lab, i in [("age < 30", te["age"] < 30), ("30–49", (te["age"] >= 30) & (te["age"] < 50)), ("50+", te["age"] >= 50)]:
    print(f"{lab:8s} n={i.sum():3d}  prevalence={te.loc[i, 'y'].mean():.2f}  AUC={roc_auc_score(te.loc[i, 'y'], te.loc[i, 'p']):.3f}")

Questions to discuss: one community, women only, data collected decades ago. Would this model be calibrated for Indian adults? Who consented, and for what? Small subgroups (21 women aged 50+) give unstable numbers.

---
# Project B — fictional data: medication adherence (linear regression)
500 patients on long-term therapy. Outcome: **PDC**, the proportion of days covered by dispensed medicine (0–100%); 80% or more is the usual adherence cut-off.

In [ ]:
adh = load("adherence_pdc.csv")
print("Mean PDC:", round(adh["pdc_percent"].mean(), 1), "| adherent (PDC ≥ 80):", round((adh["pdc_percent"] >= 80).mean(), 3))
adh.head()

In [ ]:
Xa = adh.drop(columns=["patient_id", "pdc_percent"]); ya = adh["pdc_percent"]
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=0.25, random_state=42)
lin = LinearRegression().fit(Xa_tr, ya_tr)
pred = lin.predict(Xa_te)
print("RMSE:", round(mean_squared_error(ya_te, pred) ** 0.5, 2), "points | R²:", round(r2_score(ya_te, pred), 3))
print("Baseline RMSE:", round(mean_squared_error(ya_te, np.full(len(ya_te), ya_tr.mean())) ** 0.5, 2))
pd.Series(lin.coef_, index=Xa.columns).round(3).rename("PDC points per unit")

**Interpretation:** pharmacist counselling ≈ +10 PDC points; SMS reminders ≈ +5; each extra daily dose ≈ −5. Caution: this is observational. Patients who accept counselling may differ in other ways (Unit II, confounding).

In [ ]:
# What-if for one patient: twice-daily → once-daily, plus counselling
pt = pd.DataFrame([{"age": 64, "daily_doses": 2, "pill_count": 6, "monthly_cost_rs": 900, "pharmacist_counselling": 0,
                    "sms_reminders": 0, "distance_km": 12, "lives_alone": 1}])
plan = pt.assign(daily_doses=1, pharmacist_counselling=1)
print("Predicted PDC now:      ", round(lin.predict(pt)[0], 1))
print("Once daily + counselling:", round(lin.predict(plan)[0], 1))

---
# Step 9 · Your project
Pick a topic from the syllabus list (dissolution rate, tablet hardness, QC batch failure, ADR prediction, hospital readmission, PK-based dose, diabetes risk, antibiotic success, ICU stay, adherence, disease severity…). Copy the cells below and fill each step. Present: problem → data → model → metrics → interpretation → limitations and ethics, in 8 slides or fewer.

In [ ]:
# STEP 1  Problem (one sentence) and outcome type: continuous → LinearRegression, yes/no → LogisticRegression
# STEP 2  df = load("your_file.csv") or pd.read_csv("...")
# STEP 3  Choose X (predictors) and y (outcome); explore with describe(), value_counts(), isna().mean()
# STEP 4  Preprocess: fix impossible values, encode categories with pd.get_dummies, impute inside a Pipeline
# STEP 5  train_test_split(..., random_state=42), fit the model on the training set only
# STEP 6  Evaluate on the test set: RMSE and R² (linear) or AUC, sensitivity, specificity, precision (logistic)
# STEP 7  Interpret coefficients / odds ratios in clinical units
# STEP 8  Bias and ethics: subgroups, data source, consent, who acts on the prediction
print("Template ready")

---
# Lab exercises
1. Project A: add insulin back (with imputation). Does AUC improve or worsen? Why might that be?
2. Project A: find the threshold that gives sensitivity ≥ 0.85 and report precision.
3. Project B: convert PDC to a yes/no outcome (PDC ≥ 80) and fit logistic regression. Report AUC and the odds ratio for counselling.
4. Project B: which single change would you recommend first for the what-if patient, and why?

## Solutions

In [ ]:
# Lab 1
f2 = features + ["insulin"]
m_ins = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression()).fit(pima.loc[X_tr.index, f2], y_tr)
print("AUC with insulin:", round(roc_auc_score(y_te, m_ins.predict_proba(pima.loc[X_te.index, f2])[:, 1]), 3),
      "| without:", round(roc_auc_score(y_te, p), 3))
print("Half of insulin values were imputed, so the column adds almost nothing.")

In [ ]:
# Lab 2
for t in np.arange(0.5, 0.0, -0.01):
    r = evaluate(y_te, p, round(t, 2))
    if r["sensitivity"] >= 0.85:
        print(r); break

In [ ]:
# Lab 3
yb = (adh["pdc_percent"] >= 80).astype(int)
a, b, ya2, yb2 = train_test_split(Xa, yb, test_size=0.25, random_state=42, stratify=yb)
lg = LogisticRegression(C=1e6, max_iter=10000).fit(a, ya2)
print("AUC:", round(roc_auc_score(yb2, lg.predict_proba(b)[:, 1]), 3))
print("OR for pharmacist counselling:", round(np.exp(lg.coef_[0][list(Xa.columns).index("pharmacist_counselling")]), 2))

In [ ]:
# Lab 4
for change, row in [("once daily", pt.assign(daily_doses=1)), ("counselling", pt.assign(pharmacist_counselling=1)),
                    ("SMS reminders", pt.assign(sms_reminders=1))]:
    print(f"{change:14s} → predicted PDC {lin.predict(row)[0]:.1f}")

**Data notes.** Pima: real public data (NIDDK; UCI Machine Learning Repository). Adherence: fictional teaching data.